# 说明
## 功能概述
本代码主要演示了如何在 LlamaIndex 中实现**工具调用（Tool Calling）**功能，让大语言模型能够智能地调用外部工具（如自定义函数和文档检索系统）来回答问题。这实现了：
1. 让LLM"知道何时该调用什么工具"
2. 将外部能力（计算、文档检索等）无缝集成到对话中
3. 创建能处理复杂查询的智能代理（Agent）

## 核心实现步骤
1. **定义基础工具函数**
    - 创建简单Python函数（如加法、神秘函数）
    - 用 `FunctionTool` 包装成LLM可调用的工具
2. **构建文档检索系统**
    - 加载PDF文档并分割成文本块
    - 创建向量索引（`VectorStoreIndex`）支持语义搜索
    - 创建摘要索引（`SummaryIndex`）提供文档概览
3. **创建高级查询工具**
    - 实现带元数据过滤的查询功能（如按页码检索）
    - 将查询引擎包装为 `QueryEngineTool`
4. **构建智能代理**
    - 将所有工具组合成工具列表
    - 使用 `FunctionCallingAgentWorker` 创建能自主决策的代理
    - 代理根据问题决定是否调用工具及调用哪个工具

# Lesson 2: Tool Calling

## Setup

In [1]:
from helper import get_openai_api_key, get_dashscope_api_key
import os
import nest_asyncio
# 应用异步支持（在Jupyter环境中必需）
nest_asyncio.apply()

## 1. Define a Simple Tool

In [2]:
from llama_index.core.tools import FunctionTool

# 定义基础工具函数
def add(x: int, y: int) -> int:
    """Adds two integers together.
    简单加法工具：计算两个数的和"""
    return x + y

def mystery(x: int, y: int) -> int: 
    """Mystery function that operates on top of two numbers.
    神秘函数工具：根据输入返回不同结"""
    return (x + y) * (x + y)

# 将函数包装为LLM可调用的工具
add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [3]:
from llama_index.llms.openai import OpenAI
from llama_index.llms.openai_like import OpenAILike

# 配置LLM并执行工具调用
# 配置全局设置：指定使用的语言模型和嵌入模型
# llm = OpenAI(model="gpt-3.5-turbo")

# 这里用了DashScope的大模型替代OpenAI模型，
# LlamaIndex支持多种LLM接口，DaskScope兼容OpenAI API，可以使用OpenAILike类调用
# LlamaIndex也有专门的DashScope支持包，具体见 https://developers.llamaindex.ai/python/examples/llm/dashscope/
llm = OpenAILike(
    api_key=get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)
response = llm.predict_and_call(
    [add_tool, mystery_tool], 
    "Tell me the output of the mystery function on 2 and 9", 
    verbose=True
)
print(str(response))

=== Calling Function ===
Calling function: mystery with args: {"x": 2, "y": 9}
=== Function Output ===
121
121


## 2. Define an Auto-Retrieval Tool

### Load Data

In [4]:
from llama_index.core import SimpleDirectoryReader
# 加载文档
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()
print(f"成功加载 {len(documents)} 个文档")

2026-05-05 16:11:23,207 - INFO - NumExpr defaulting to 10 threads.


成功加载 29 个文档


In [5]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)
print(f"文档已分割成 {len(nodes)} 个文本块")

文档已分割成 34 个文本块


In [9]:
print(nodes[0].get_content(metadata_mode="all"))

page_label: 1
file_name: metagpt.pdf
file_path: metagpt.pdf
file_type: application/pdf
file_size: 16911937
creation_date: 2025-10-20
last_modified_date: 2025-10-20

Preprint
METAGPT: M ETA PROGRAMMING FOR A
MULTI-AGENT COLLABORATIVE FRAMEWORK
Sirui Hong1∗, Mingchen Zhuge2∗, Jonathan Chen1, Xiawu Zheng3, Yuheng Cheng4,
Ceyao Zhang4, Jinlin Wang1, Zili Wang, Steven Ka Shing Yau5, Zijuan Lin4,
Liyang Zhou6, Chenyu Ran1, Lingfeng Xiao1,7, Chenglin Wu1†, J¨urgen Schmidhuber2,8
1DeepWisdom, 2AI Initiative, King Abdullah University of Science and Technology,
3Xiamen University, 4The Chinese University of Hong Kong, Shenzhen,
5Nanjing University, 6University of Pennsylvania,
7University of California, Berkeley, 8The Swiss AI Lab IDSIA/USI/SUPSI
ABSTRACT
Remarkable progress has been made on automated problem solving through so-
cieties of agents based on large language models (LLMs). Existing LLM-based
multi-agent systems can already solve simple dialogue tasks. Solutions to more
complex tasks,

In [7]:
from llama_index.core import VectorStoreIndex
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = llm
# 需要openai key所以改成开源模型
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
model_real_path = os.path.expanduser(
    "~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
Settings.embed_model = HuggingFaceEmbedding(
        model_name=model_real_path,
        # 默认情况下，LlamaIndex 会尝试自动下载和加载模型
        device="cpu",  # 如果您没有GPU，可以使用"cpu"
        # 连不了外网下载模型到本地的记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
        local_files_only=True,
    )

# 创建两索引 VectorStoreIndex：用于语义搜索，查找与问题最相关的文本块
vector_index = VectorStoreIndex(nodes)
# 创建查询引擎 query_engine：基于向量索引，适合回答具体细节问题
query_engine = vector_index.as_query_engine(similarity_top_k=2)
print("已创建索引和查询引擎")

2026-05-05 16:12:26,431 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a


已创建索引和查询引擎


In [10]:
from llama_index.core.vector_stores import MetadataFilters

# 创建过滤器，只搜索指定页码的内容
# page_label: 1
# file_name: metagpt.pdf
# file_path: metagpt.pdf
# file_type: application/pdf
# file_size: 16911937
# creation_date: 2025-10-20
# last_modified_date: 2025-10-20
# 这些元数据是SimpleDirectoryReader在加载PDF时自动提取的，
# 特别是page_label是由PDF解析器添加的，表示该文本块来自PDF的哪一页。

# 关于查询执行顺序的问题： 在LlamaIndex中，filters参数是在创建查询引擎时指定的，它会影响向量检索的过程。具体来说：
# 1、当使用vector_index.as_query_engine(filters=...)时，过滤器会在向量检索过程中应用
# 2、向量检索会先计算所有节点的相似度，然后应用过滤器只保留匹配的节点
# 3、最后返回过滤后最相似的top_k个结果
# LlamaIndex也支持另一种方式：先过滤再检索。这取决于你如何配置查询引擎和检索器
# 如果使用VectorIndexRetriever检索器，则先过滤再检索
query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "2"}
        ]
    )
)

# 测试请求
response = query_engine.query(
    "What are some high-level results of MetaGPT?",  # 
)

print(str(response))

2026-05-05 16:15:34,134 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


MetaGPT has achieved notable results, particularly in code generation benchmarks. It has set a new state-of-the-art with 85.9% and 87.7% in Pass@1 on the HumanEval and MBPP datasets, respectively. Additionally, MetaGPT has demonstrated a 100% task completion rate in experimental evaluations, showing its robustness and efficiency in terms of time and token costs. These outcomes highlight its capability to handle higher levels of software complexity and offer extensive functionality compared to other popular frameworks.


In [11]:
# 查看引用了多少个文档片段
print(len(response.source_nodes))
# 查看引用片段的元信息
for n in response.source_nodes:
    print(n.metadata)

1
{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}


### Define the Auto-Retrieval Tool（定义自动检索工具）

In [12]:
from typing import List  # 导入类型注解工具，用于指定变量类型
from llama_index.core.vector_stores import FilterCondition  # 导入过滤条件类，用于文档检索过滤


def vector_query(
    query: str,  # 用户的查询问题，例如"MetaGPT的主要结果是什么？"
    page_numbers: List[str]  # 要查询的页码列表，例如["1", "2", "3"]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    执行向量搜索，从文档中查找与查询相关的内容
    
    参数:
    query (str): 用户输入的查询字符串，会被转换为向量进行语义搜索
    page_numbers (List[str]): 要过滤的页码列表。留空表示搜索所有页面。
                              例如: ["2", "3"] 表示只搜索第2页和第3页的内容
    
    返回:
    str: 搜索结果的文本内容
    
    工作原理:
    1. 创建元数据过滤器，只搜索指定页码的内容
    2. 使用向量索引执行语义搜索（理解意思相近的内容）
    3. 返回最相关的搜索结果
    """

    # 创建元数据字典列表，用于过滤特定页码
    # 例如: [{"key": "page_label", "value": "2"}, {"key": "page_label", "value": "3"}]
    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]
    
    # 创建查询引擎，配置搜索参数
    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    # 执行查询并获取响应
    response = query_engine.query(query)
    # 返回响应内容（字符串形式）
    return response
    
# 将vector_query函数包装为LLM可调用的工具
vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [13]:
# 调用工具处理查询
response = llm.predict_and_call(
    [vector_query_tool],   # 工具列表，这里只使用vector_query_tool
    "What are the high-level results of MetaGPT as described on page 2?",   # MetaGPT 在第2页描述的高级结果有哪些？
    verbose=True  # 开启详细日志，可以看到LLM的思考过程
)
# 查看引用了多少个文档片段
print(len(response.source_nodes))
# 查看引用片段的元信息
for n in response.source_nodes:
    print(n.metadata)

2026-05-05 16:19:03,010 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: vector_tool with args: {"query": "high-level results of MetaGPT", "page_numbers": ["2"]}


2026-05-05 16:19:10,333 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT has demonstrated impressive performance in code generation benchmarks, achieving state-of-the-art results with 85.9% and 87.7% in Pass@1 on the HumanEval and MBPP datasets, respectively. Additionally, it has shown a 100% task completion rate in experimental evaluations, highlighting its robustness and efficiency in handling complex software projects. These outcomes underscore MetaGPT's effectiveness in automatic requirement analysis, system design, code generation, modification, execution, and debugging during runtime.
1
{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}


## Let's add some other tools!

In [14]:
from llama_index.core import SummaryIndex  # 用于创建整个文档的摘要索引
from llama_index.core.tools import QueryEngineTool  # 将查询引擎包装为LLM可调用的工具

# 创建文档摘要索引（就像给整本书写一个简短的概述）
# SummaryIndex 会分析所有文本块，生成一个连贯的文档摘要
summary_index = SummaryIndex(nodes)

# 基于摘要索引创建查询引擎
# 这个引擎能回答关于整个文档的概括性问题
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",  # 摘要生成模式
    use_async=True,  # 使用异步处理，提高效率
)

# 将查询引擎包装为LLM可调用的工具
# 这样LLM就能"知道"它可以使用这个工具来获取文档摘要
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",  # 工具名称，LLM会用这个名字识别工具
    query_engine=summary_query_engine,  # 实际的查询引擎
    description=(
        "Useful if you want to get a summary of MetaGPT"  # 工具描述，告诉LLM什么情况下使用
    ),
)


In [15]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What are the MetaGPT comparisons with ChatDev described on page 8?",   # MetaGPT 在第8页与ChatDev的比较有哪些？
    verbose=True
)

for n in response.source_nodes:
    print(n.metadata)


2026-05-05 16:21:55,734 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: vector_tool with args: {"query": "MetaGPT comparisons with ChatDev", "page_numbers": ["8"]}


2026-05-05 16:22:07,317 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT shows superior performance compared to ChatDev across several key metrics. In terms of executability, MetaGPT scores 3.75, which is notably higher and closer to a perfect score of 4. It also completes tasks faster, taking only 503 seconds compared to ChatDev's 762 seconds. While MetaGPT uses more tokens overall (24,613 or 31,255) than ChatDev (19,292), it is more efficient in token usage per line of code, requiring only about 126.5/124.3 tokens, whereas ChatDev needs 248.9 tokens. Additionally, MetaGPT generates more lines of code and requires significantly less human revision, with a cost of 0.83 compared to ChatDev's 2.5. These metrics indicate that MetaGPT is more effective and efficient in software development tasks.
{'page_label': '8', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}


In [16]:
# 以下代码如果LLM决策不需要掉工具就会报错
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What is a summary of the paper?",  # 这篇论文的摘要是什么？
    verbose=True,
    error_on_no_tool_call=False
)
for n in response.source_nodes:
    print(n.metadata)

2026-05-05 16:22:26,585 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: vector_tool with args: {"query": "main content of the paper", "page_numbers": []}


2026-05-05 16:22:37,746 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


=== Function Output ===
The main content of the paper focuses on the software development process within MetaGPT, which heavily relies on Standard Operating Procedures (SOPs). It details a workflow where user requirements are analyzed by a Product Manager, who then creates a detailed Product Requirements Document (PRD) that includes User Stories and a Requirement Pool. This document is used by the Architect to design system components such as File Lists, Data Structures, and Interface Definitions. The Project Manager then distributes tasks based on this design, and Engineers develop the specified classes and functions. QA Engineers create test cases to ensure code quality, leading to the final production of the software solution.

Additionally, the paper discusses the communication protocol in MetaGPT, questioning the sufficiency of using only natural language for complex task-solving in multi-agent frameworks. It also provides a table with additional results from using MetaGPT without